# Optimization Note 2: Line Search and Quasi-Newton (BFGS)

**Goal:** Fix Newton's two problems — divergence from bad starting points and the need for an exact Hessian.

## Problem 1: Globalization via Line Search

Pure Newton takes the full step $x_{k+1} = x_k + d_k$. This can increase $f$.

**Fix:** Take a **damped step** $x_{k+1} = x_k + \alpha_k d_k$ where $\alpha_k \in (0, 1]$ is chosen to guarantee sufficient decrease.

### The Armijo Condition

Accept step length $\alpha$ if:

$$f(x_k + \alpha d_k) \leq f(x_k) + c_1 \alpha \nabla f(x_k)^T d_k$$

where $c_1 \in (0, 1)$ is small (typically $10^{-4}$). This says: $f$ must decrease by at least a fraction of the predicted decrease.

### Backtracking

Start with $\alpha = 1$ (full Newton step). If Armijo fails, try $\alpha/2$, then $\alpha/4$, etc.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=8, suppress=True)

In [ ]:
def backtracking_line_search(f, x, d, g, c1=1e-4, rho=0.5, max_bt=40):
    """Backtracking line search with Armijo condition.
    
    Returns step length alpha.
    """
    alpha = 1.0
    f0 = f(x)
    slope = g @ d  # directional derivative
    
    for _ in range(max_bt):
        if f(x + alpha * d) <= f0 + c1 * alpha * slope:
            return alpha
        alpha *= rho
    
    return alpha  # return whatever we have

In [ ]:
def newton_line_search(f, grad, hess, x0, tol=1e-10, max_iter=100):
    """Newton's method with backtracking line search."""
    x = np.array(x0, dtype=float)
    history = [{'x': x.copy(), 'f': f(x), 'grad_norm': np.linalg.norm(grad(x)), 'alpha': 1.0}]
    
    for k in range(max_iter):
        g = grad(x)
        if np.linalg.norm(g) < tol:
            break
        
        H = hess(x)
        
        # Hessian modification: if not positive definite, add multiple of I
        eigvals = np.linalg.eigvalsh(H)
        if min(eigvals) < 1e-8:
            shift = abs(min(eigvals)) + 1e-4
            H = H + shift * np.eye(len(x))
        
        d = np.linalg.solve(H, -g)
        alpha = backtracking_line_search(f, x, d, g)
        x = x + alpha * d
        history.append({'x': x.copy(), 'f': f(x), 'grad_norm': np.linalg.norm(grad(x)), 'alpha': alpha})
    
    return x, history

In [ ]:
# Rosenbrock — the case that made pure Newton diverge
def rosenbrock(x):
    return 100 * (x[1] - x[0]**2)**2 + (1 - x[0])**2

def rosenbrock_grad(x):
    return np.array([
        -400 * x[0] * (x[1] - x[0]**2) - 2 * (1 - x[0]),
        200 * (x[1] - x[0]**2)
    ])

def rosenbrock_hess(x):
    return np.array([
        [1200 * x[0]**2 - 400 * x[1] + 2, -400 * x[0]],
        [-400 * x[0], 200]
    ])

# Start far away
x0 = np.array([-5.0, 5.0])
x_opt, hist = newton_line_search(rosenbrock, rosenbrock_grad, rosenbrock_hess, x0)

print(f"Converged in {len(hist)-1} iterations")
print(f"Solution: x = {x_opt}")
print(f"f(x*) = {rosenbrock(x_opt):.2e}")
print()
print(f"{'Iter':>4} {'f(x)':>14} {'||grad||':>12} {'alpha':>8}")
print("-" * 42)
for i, h in enumerate(hist[:15]):
    print(f"{i:>4} {h['f']:>14.6e} {h['grad_norm']:>12.4e} {h['alpha']:>8.4f}")
if len(hist) > 15:
    print(f"  ... ({len(hist)-1} total iterations)")

## Problem 2: Avoiding the Hessian with BFGS

Computing $\nabla^2 f$ is expensive ($O(n^2)$ storage, $O(n^2)$ computation). **BFGS** (Broyden-Fletcher-Goldfarb-Shanno) builds an approximation $B_k \approx \nabla^2 f(x_k)$ using only gradient information.

### The BFGS Update

After step $k$, define:
- $s_k = x_{k+1} - x_k$ (the step)
- $y_k = \nabla f(x_{k+1}) - \nabla f(x_k)$ (gradient change)

The **secant condition** requires $B_{k+1} s_k = y_k$ (the approximation matches the actual curvature along the step direction).

The BFGS update:

$$B_{k+1} = B_k - \frac{B_k s_k s_k^T B_k}{s_k^T B_k s_k} + \frac{y_k y_k^T}{y_k^T s_k}$$

This is a rank-2 update — cheap at $O(n^2)$, no need to ever form the true Hessian.

In [ ]:
def bfgs(f, grad, x0, tol=1e-8, max_iter=200):
    """BFGS quasi-Newton method with backtracking line search."""
    n = len(x0)
    x = np.array(x0, dtype=float)
    B = np.eye(n)  # initial Hessian approximation
    g = grad(x)
    history = [{'x': x.copy(), 'f': f(x), 'grad_norm': np.linalg.norm(g)}]
    
    for k in range(max_iter):
        if np.linalg.norm(g) < tol:
            break
        
        # Search direction
        d = np.linalg.solve(B, -g)
        
        # Line search
        alpha = backtracking_line_search(f, x, d, g)
        
        # Take step
        s = alpha * d
        x_new = x + s
        g_new = grad(x_new)
        y = g_new - g
        
        # BFGS update (skip if curvature condition fails)
        sy = s @ y
        if sy > 1e-12:
            Bs = B @ s
            B = B - np.outer(Bs, Bs) / (s @ Bs) + np.outer(y, y) / sy
        
        x = x_new
        g = g_new
        history.append({'x': x.copy(), 'f': f(x), 'grad_norm': np.linalg.norm(g)})
    
    return x, history

In [ ]:
# Compare Newton (with line search) vs BFGS on Rosenbrock
x0 = np.array([-1.0, 1.0])

x_newton, hist_newton = newton_line_search(rosenbrock, rosenbrock_grad, rosenbrock_hess, x0)
x_bfgs, hist_bfgs = bfgs(rosenbrock, rosenbrock_grad, x0)

print(f"Newton + line search: {len(hist_newton)-1} iterations, f = {rosenbrock(x_newton):.2e}")
print(f"BFGS:                 {len(hist_bfgs)-1} iterations, f = {rosenbrock(x_bfgs):.2e}")
print(f"\nBFGS uses more iterations but NO Hessian evaluations.")
print(f"Each Newton iteration: 1 Hessian + 1 gradient + 1 O(n^3) solve")
print(f"Each BFGS iteration:   0 Hessians + 1 gradient + 1 O(n^2) update + 1 O(n^2) solve")

In [ ]:
# Visualize both paths
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, hist, title in [(axes[0], hist_newton, f'Newton + LS ({len(hist_newton)-1} iters)'),
                         (axes[1], hist_bfgs, f'BFGS ({len(hist_bfgs)-1} iters)')]:
    xx = np.linspace(-1.5, 1.5, 200)
    yy = np.linspace(-0.5, 2.0, 200)
    X, Y = np.meshgrid(xx, yy)
    Z = 100 * (Y - X**2)**2 + (1 - X)**2
    ax.contour(X, Y, Z, levels=np.logspace(-1, 3, 15), cmap='viridis', alpha=0.5)
    
    path = np.array([h['x'] for h in hist])
    ax.plot(path[:, 0], path[:, 1], 'ro-', markersize=3, linewidth=1)
    ax.plot(1, 1, 'k*', markersize=15)
    ax.plot(path[0, 0], path[0, 1], 'gs', markersize=10)
    ax.set_title(title)
    ax.set_xlabel('$x_1$')
    ax.set_ylabel('$x_2$')

plt.tight_layout()
plt.show()

## L-BFGS: Limited-Memory BFGS

BFGS stores the full $n \times n$ matrix $B_k$ — this is $O(n^2)$ memory. For large-scale problems, **L-BFGS** stores only the last $m$ curvature pairs $(s_k, y_k)$ and computes $B_k^{-1} g$ implicitly.

- Memory: $O(mn)$ instead of $O(n^2)$, with $m$ typically 5-20
- Cost per iteration: $O(mn)$ instead of $O(n^2)$
- Convergence: superlinear (not quite quadratic like Newton)

ripopt uses L-BFGS as a **fallback** (`hessian_approximation_lbfgs` option) when exact Hessians aren't available. It stores up to 10 curvature pairs.

In [ ]:
def lbfgs(f, grad, x0, m=10, tol=1e-8, max_iter=200):
    """L-BFGS with backtracking line search."""
    n = len(x0)
    x = np.array(x0, dtype=float)
    g = grad(x)
    history = [{'x': x.copy(), 'f': f(x), 'grad_norm': np.linalg.norm(g)}]
    
    s_list = []  # stores recent s_k
    y_list = []  # stores recent y_k
    rho_list = []  # stores 1 / (y_k @ s_k)
    
    for k in range(max_iter):
        if np.linalg.norm(g) < tol:
            break
        
        # Two-loop recursion to compute d = -H_k @ g
        q = g.copy()
        alphas = []
        for s, y, rho in reversed(list(zip(s_list, y_list, rho_list))):
            a = rho * (s @ q)
            alphas.append(a)
            q -= a * y
        alphas.reverse()
        
        # Initial Hessian scaling
        if len(s_list) > 0:
            gamma = (s_list[-1] @ y_list[-1]) / (y_list[-1] @ y_list[-1])
            r = gamma * q
        else:
            r = q.copy()
        
        for (s, y, rho), a in zip(zip(s_list, y_list, rho_list), alphas):
            b = rho * (y @ r)
            r += (a - b) * s
        
        d = -r
        
        # Line search
        alpha = backtracking_line_search(f, x, d, g)
        
        s = alpha * d
        x_new = x + s
        g_new = grad(x_new)
        y = g_new - g
        
        sy = s @ y
        if sy > 1e-12:
            s_list.append(s)
            y_list.append(y)
            rho_list.append(1.0 / sy)
            if len(s_list) > m:
                s_list.pop(0)
                y_list.pop(0)
                rho_list.pop(0)
        
        x = x_new
        g = g_new
        history.append({'x': x.copy(), 'f': f(x), 'grad_norm': np.linalg.norm(g)})
    
    return x, history

In [ ]:
# Compare all three on a larger Rosenbrock (n=10)
def rosenbrock_nd(x):
    return sum(100 * (x[i+1] - x[i]**2)**2 + (1 - x[i])**2 for i in range(len(x)-1))

def rosenbrock_nd_grad(x):
    n = len(x)
    g = np.zeros(n)
    for i in range(n - 1):
        g[i] += -400 * x[i] * (x[i+1] - x[i]**2) - 2 * (1 - x[i])
        g[i+1] += 200 * (x[i+1] - x[i]**2)
    return g

n = 10
x0 = np.zeros(n)

x_bfgs, hist_bfgs = bfgs(rosenbrock_nd, rosenbrock_nd_grad, x0)
x_lbfgs, hist_lbfgs = lbfgs(rosenbrock_nd, rosenbrock_nd_grad, x0, m=10)

print(f"n = {n}  (optimum: all ones)")
print(f"BFGS:   {len(hist_bfgs)-1:>4} iters, f = {rosenbrock_nd(x_bfgs):.2e}, ||g|| = {hist_bfgs[-1]['grad_norm']:.2e}")
print(f"L-BFGS: {len(hist_lbfgs)-1:>4} iters, f = {rosenbrock_nd(x_lbfgs):.2e}, ||g|| = {hist_lbfgs[-1]['grad_norm']:.2e}")
print(f"\nBFGS memory:  O(n^2) = {n*n} entries")
print(f"L-BFGS memory: O(mn) = {10*n} entries  ({10*n/(n*n)*100:.0f}% of BFGS)")

## What We've Learned

1. **Line search** (backtracking + Armijo) prevents divergence by controlling step length
2. **Hessian modification** (adding $\delta I$) handles indefinite Hessians
3. **BFGS** approximates the Hessian from gradient differences — no Hessian evaluations needed
4. **L-BFGS** uses $O(mn)$ memory instead of $O(n^2)$ — practical for large $n$
5. Trade-off: more iterations but cheaper per iteration

## What's Next

So far we've solved **unconstrained** problems. Real optimization problems have **constraints**: variable bounds ($x \geq 0$), linear constraints, nonlinear constraints. In Note 3, we introduce the **logarithmic barrier method** — the key idea that turns a constrained problem into a sequence of unconstrained ones.

---

*This is Optimization Note 2 in a series building from Newton's method to the interior point optimizer [ripopt](https://github.com/jkitchin/ripopt).*